# 02 - RWGAN: a connection-count reward

**Where this sits:** WGAN-GP -> **`RWGAN`**. Same WGAN-GP core as notebook `01`,
plus one **frozen** CNN reward network that regresses a topology's normalized
connection count. Its MSE against a target is blended into the generator loss
with weight `(1 - LAMBDA)`, and `LAMBDA` is annealed downward during training so
control hands over from the critic to the reward.

## The reward term

The generator loss is

```
L_G = LAMBDA * (-E[critic(G(z))])  +  (1 - LAMBDA) * MSE(reward(G(z)), REWARD_TARGET)
```

(`gannoc.losses.make_generator_wasserstein_loss` / `make_reward_mse_loss`).
`REWARD_TARGET` lives in the reward net's normalized `[-1, 1]` space -- the same
space as `gannoc.data.NoCDataset.connections_norm`, where `-1` is 8 links and
`+1` is 18 links. A target above `0.5` asks for denser topologies.

## The LAMBDA anneal

`gannoc.losses.LambdaSchedule` starts `LAMBDA` at `1.0` (pure WGAN-GP) and, from
`LAMBDA_START_EPOCH` on, subtracts `LAMBDA_DELTA` every `LAMBDA_EVERY_N_EPOCHS`
epochs until it reaches the floor. Those are module constants in
`gannoc/training.py`, set to the paper's long-run schedule (annealing starts at
epoch 100 of 250). This demo assigns shorter values so `LAMBDA` actually moves
within 8 epochs.

## Setup

In [ ]:
import sys
sys.path.insert(0, "../src")

import pickle
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from gannoc import data as gdata
from gannoc import training
from gannoc.reward_network import load_reward_network
from gannoc.evaluate import evaluate_generator

OUT = "output"
REWARD_CKPT = "../data/reference/reward_networks/r_cnn_214.h5"

## 1. Training set + reward network

Reuse the tiny dataset from notebook `01` (regenerated here so this notebook
stands alone). For the reward network we load the shipped, pretrained
`r_cnn_214.h5` -- the checkpoint the reference RWGAN implementation
(`RWGANgp_pretrainedR.py`) loaded (~2.19M params).
`gannoc.training.train_rwgan` loads it frozen (`trainable=False`) from the
`reward_checkpoint` path; we load it here too just to sanity-check it.

Note its architecture is not the critic-shaped one the paper describes for the
reward (`gannoc.reward_network.build_reward_net`) -- it is
`build_reward_net_cnn2`, one of the variants the original work explored. To
train a fresh, critic-shaped reward net instead, use
`gannoc.reward_training.train` and point `reward_checkpoint` at its saved
`.h5`.

In [ ]:
raw = gdata.generate_dataset(connection_counts=range(8, 19), samples_per_class=200, seed=0)
gdata.save_dataset(raw, f"{OUT}/_demo_nocs.npz")
dataset = gdata.load_dataset(f"{OUT}/_demo_nocs.npz")
print(f"{len(dataset)} topologies")

reward = load_reward_network(REWARD_CKPT)  # tf.keras.models.load_model(..., compile=False), trainable=False
reward.summary()

# sanity check: predicted vs true normalized connection count on real topologies
pred = reward.predict(dataset.images()[:200], verbose=0).ravel()
true = dataset.connections_norm()[:200].ravel()
print("reward MAE on real topologies:", float(np.mean(np.abs(pred - true))))

## 2. Train the RWGAN

`lambda_floor=0.5` with the shortened anneal takes `LAMBDA` from `1.0` down to
`0.5` over the run, so by the last epochs the reward MSE carries half the
generator gradient. `reward_target=0.7` asks for topologies well above the
midpoint of the 8..18 link range.

In [ ]:
# Demo overrides of gannoc.training's module constants (paper values otherwise).
training.SEED = 0
training.EVAL_EVERY = 8
training.REWARD_TARGET = 0.7        # normalized [-1, 1]; > 0.5 => denser topologies
training.LAMBDA_START_EPOCH = 1     # paper: 100 -- begin annealing at epoch 1 here
training.LAMBDA_DELTA = -0.1        # paper: -0.005 -- LAMBDA -= 0.1 ...
training.LAMBDA_EVERY_N_EPOCHS = 1  # ... every epoch, until the floor

result = training.train_rwgan(
    dataset_path=f"{OUT}/_demo_nocs.npz",
    reward_checkpoint=REWARD_CKPT,
    output_dir=OUT,
    run_name="02_rwgan",
    epochs=8,
    lambda_floor=0.5,   # LAMBDA anneals 1.0 -> 0.5
)
with open(result["history"], "rb") as fh:
    history = pickle.load(fh)
print("final LAMBDA:", history.get("lambda", [None])[-1])

generator = tf.keras.models.load_model(result["generator_checkpoint"], compile=False)
metrics = evaluate_generator(generator, dataset, n_samples=1000, seed=0)
print(metrics.summary())

## 3. Connection-count distribution vs the WGAN-GP reference

Compare the RWGAN's generated topologies against the bundled WGAN-GP set
`data/reference/generated_topologies/wgan_valid_topologies.pkl` (446 valid
topologies, mean ~12 links). The RWGAN histogram should sit to the right.

In [ ]:
# RWGAN: sample the freshly trained generator
latent_dim = generator.input_shape[-1]
z = np.random.default_rng(1).random((600, latent_dim)).astype("float32")  # uniform, as in training
gen = generator.predict(z, verbose=0)[..., 0]
gen_mats = [gdata.binarize_symmetric(m, threshold=0.0) for m in gen]
rwgan_conn = np.array([gdata.n_connections_of(m) for m in gen_mats
                       if gdata.is_valid_topology(m)])

# WGAN-GP reference: the bundled paper set
with open("../data/reference/generated_topologies/wgan_valid_topologies.pkl", "rb") as fh:
    wgan_ref = np.asarray(pickle.load(fh))
wgan_conn = wgan_ref.sum(axis=(1, 2)) // 2

print(f"WGAN-GP reference mean : {wgan_conn.mean():.2f} links  (n={len(wgan_conn)})")
print(f"RWGAN generated mean   : {rwgan_conn.mean():.2f} links  (n={len(rwgan_conn)})")

bins = np.arange(7.5, 19.5, 1)
plt.figure(figsize=(7, 4))
plt.hist(wgan_conn, bins=bins, alpha=0.6, label=f"WGAN-GP ref (mean {wgan_conn.mean():.1f})")
plt.hist(rwgan_conn, bins=bins, alpha=0.6, label=f"RWGAN (mean {rwgan_conn.mean():.1f})")
plt.xlabel("physical connections")
plt.ylabel("count")
plt.title("Connection-count distribution: RWGAN vs WGAN-GP")
plt.legend()
plt.tight_layout()
plt.savefig(f"{OUT}/02_connection_histogram.png", dpi=120)
plt.show()

## Result

As `LAMBDA` anneals, the frozen reward network's target connection count starts
steering the generator: the RWGAN's generated topologies carry more links on
average than the WGAN-GP baseline. In the paper this denser bias is what
lowers packet latency under uniform traffic -- measuring that needs an external
cycle-accurate NoC simulator and is out of scope here. Pushing `reward_target`
higher or `lambda_floor` lower strengthens the effect, at the cost of validity
as the degree-<=4 constraint gets harder to satisfy.